<a href="https://colab.research.google.com/github/shreyanshxt/Autism-Detection/blob/main/Hand_writing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

LOCAL_DATA = '/content/handwriting'
os.makedirs(LOCAL_DATA, exist_ok=True)

!unzip -q "/content/drive/MyDrive/Handwriting/Handwritting-20260202T093629Z-3-001.zip" -d {LOCAL_DATA}/Handwriting/

print("Extraction complete. Data is ready for preprocessing.")

Extraction complete. Data is ready for preprocessing.


In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
from torchvision import models
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Define device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path to dataset
BASE_DIR = "/content/handwriting/Handwriting/Handwritting"

# Constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42

def create_metadata_dataframe(base_dir):
    """
    Create a dataframe with metadata for all images in the dataset
    """
    data = []

    # First level: ASD, ASD with CD, Non-ASD
    for condition in os.listdir(base_dir):
        condition_path = os.path.join(base_dir, condition)
        if not os.path.isdir(condition_path):
            continue

        # Second level: Mild, Moderate, Severe
        for severity in os.listdir(condition_path):
            severity_path = os.path.join(condition_path, severity)
            if not os.path.isdir(severity_path):
                continue

            # Third level: Age groups
            for age_group in os.listdir(severity_path):
                age_path = os.path.join(severity_path, age_group)
                if not os.path.isdir(age_path):
                    continue

                # Fourth level: Activity types
                for activity in os.listdir(age_path):
                    activity_path = os.path.join(age_path, activity)
                    if not os.path.isdir(activity_path):
                        continue

                    # Get all image files
                    for img_file in os.listdir(activity_path):
                        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                            img_path = os.path.join(activity_path, img_file)

                            # Add metadata for this image
                            data.append({
                                'filepath': img_path,
                                'condition': condition,
                                'severity': severity,
                                'age_group': age_group,
                                'activity': activity,
                                'filename': img_file
                            })

    # Create DataFrame
    df = pd.DataFrame(data)

    # Extract numeric age range for easier analysis
    df['age_min'] = df['age_group'].str.extract(r'(\d+)').astype(float)

    # Add binary label for autism vs non-autism
    df['has_autism'] = df['condition'].apply(lambda x: 'no' if x == 'Non-ASD' else 'yes')

    # Normalize activity names (fix spelling variations)
    activity_mapping = {
        'Handwriitng': 'Handwriting',
        'Handwritng': 'Handwriting',
        'handwriting': 'Handwriting'
    }
    df['activity'] = df['activity'].replace(activity_mapping)

    return df

# Custom Dataset Class
class ASDDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

        # Encode labels
        self.label_encoder = LabelEncoder()
        self.data['encoded_label'] = self.label_encoder.fit_transform(self.data['has_autism'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['filepath']
        label = self.data.iloc[idx]['encoded_label']

        # Load image
        from PIL import Image
        image = Image.open(img_path).convert('RGB')

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label

# Define Transformations
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create Metadata DataFrame
metadata_df = create_metadata_dataframe(BASE_DIR)

# Split the data
train_df, test_df = train_test_split(metadata_df, test_size=0.2, random_state=SEED, stratify=metadata_df['has_autism'])

# Create Datasets
train_dataset = ASDDataset(train_df, transform=transform)
test_dataset = ASDDataset(test_df, transform=transform)

# Create Data Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Load Pretrained ConvNeXt (originally commented as ResNet-18)
model = models.convnext_tiny(weights="IMAGENET1K_V1")

# Get number of classes from the dataset
num_classes = len(train_dataset.label_encoder.classes_)

# Replace classifier head
num_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(num_features, num_classes) # Corrected: use num_classes variable as out_features

# Remove the incorrect modification for model.fc as ConvNeXt models use model.classifier
# model.fc = nn.Linear(model.fc.in_features, num_classes)

# Move model to GPU if available
model = model.to(device)

# Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward Pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(train_loader):.4f}")

# Evaluate Model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Save Model
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder': train_dataset.label_encoder
}, "resnet18_asd_gestures.pth")

# Additional Logging
print("\nDataset Information:")
print(f"Total Images: {len(metadata_df)}")
print(f"Training Images: {len(train_df)}")
print(f"Testing Images: {len(test_df)}")
print("\nClass Distribution:")
print(metadata_df['has_autism'].value_counts(normalize=True))

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 176MB/s] 


Starting Training...
Epoch [1/30], Loss: 0.7552
Epoch [2/30], Loss: 0.6451
Epoch [3/30], Loss: 0.6233
